# Topic 3D: k-Nearest Neighbors for Peer-Group Analysis
**Module 1 - Introduction to Machine Learning in Python**


In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt


## 1. Prepare Data


In [ ]:
np.random.seed(42)
n = 5000
df = pd.DataFrame({
    'income': np.random.lognormal(11, 0.7, n),
    'credit_score': np.random.normal(700, 50, n).clip(300, 850),
    'dti': np.random.uniform(0, 45, n),
    'loan_amnt': np.random.lognormal(9.5, 0.5, n),
})
logit = (-3 - 0.00001*df['income'] - 0.005*df['credit_score'] + 0.05*df['dti'] + 0.00005*df['loan_amnt'] + np.random.normal(0, 0.5, n))
from scipy.special import expit
df['default'] = (np.random.random(n) < expit(logit)).astype(int)

features = ['income', 'credit_score', 'dti', 'loan_amnt']
X = df[features]; y = df['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# CRITICAL: Scale features for k-NN
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


## 2. Choosing k with Cross-Validation


In [ ]:
k_values = [1, 3, 5, 7, 11, 21, 31, 51, 101]
aurocs = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    scores = cross_val_score(knn, X_train_s, y_train, cv=5, scoring='roc_auc')
    aurocs.append(scores.mean())
    print(f'k={k:>3}: CV AUROC = {scores.mean():.4f} (+/- {scores.std():.4f})')

best_k = k_values[np.argmax(aurocs)]
print(f'\nBest k: {best_k}')

plt.figure(figsize=(8, 4))
plt.plot(k_values, aurocs, 'o-', color='steelblue')
plt.xlabel('k (number of neighbors)')
plt.ylabel('CV AUROC')
plt.title('k-NN: AUROC vs k')
plt.axvline(x=best_k, color='red', linestyle='--')
plt.tight_layout()
plt.show()


## 3. Peer-Group Analysis for Individual Applicants


In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=10, metric='euclidean')
nn.fit(X_train_s)

# Pick 3 test applicants
for i in [0, 50, 100]:
    applicant = X_test_s[i].reshape(1, -1)
    distances, indices = nn.kneighbors(applicant)
    
    neighbor_defaults = y_train.iloc[indices[0]]
    peer_default_rate = neighbor_defaults.mean()
    actual = y_test.iloc[i]
    
    print(f'Applicant {i}:')
    print(f'  Features: {dict(zip(features, X_test.iloc[i].round(0)))}')
    print(f'  10 nearest neighbors default rate: {peer_default_rate:.2%}')
    print(f'  Actual outcome: {"Defaulted" if actual == 1 else "Repaid"}')
    print()
